In [3]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.3 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [16]:
import os, inspect, torch, torch.nn as nn, torch.nn.functional as F
import soundfile as sf, torchaudio
import pandas as pd
import numpy as np
from transformers import AutoConfig, AutoModel, AutoFeatureExtractor
from peft import LoraConfig, get_peft_model
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os, inspect, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoConfig, AutoModel, AutoFeatureExtractor
from peft import LoraConfig, get_peft_model

MID = "google/embeddinggemma-2"
SR = 16_000                              # rate the feature extractor is told; samples go in unchanged
WIN = 5120                               # window length in samples (inputs are (B, WIN))
NUM_CLASSES = 5
PROJ_DIM = 128
torch.set_num_threads(os.cpu_count())

# ---- keep only the audio tower (no vision, no text) ----
cfg = AutoConfig.from_pretrained(MID)
cfg.vision_config = None
base = AutoModel.from_pretrained(MID, config=cfg, torch_dtype=torch.float32)
audio_tower = next(m for m in base.modules() if type(m).__name__.endswith("AudioModel"))
del base


feature_extractor = AutoFeatureExtractor.from_pretrained(MID)
TOWER_TAKES_MASK = len(inspect.signature(audio_tower.forward).parameters) > 1

PROJ = {"q_proj", "k_proj", "v_proj", "o_proj"}
targets = [n for n, m in audio_tower.named_modules()
           if isinstance(m, nn.Linear) and PROJ & set(n.split("."))]
audio_tower = get_peft_model(audio_tower, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                                     target_modules=targets))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def run_tower(tower, x):
    dev = next(tower.parameters()).device
    f = feature_extractor(list(x.detach().cpu().numpy()), sampling_rate=SR, return_tensors="pt")
    feats = f["input_features"].to(dev)
    out = (tower(feats, f["input_features_mask"].to(dev))
           if TOWER_TAKES_MASK else tower(feats))
    return out[0] if isinstance(out, tuple) else out.last_hidden_state


class WDPModel(nn.Module):
    def __init__(self, tower, num_classes, proj_dim=PROJ_DIM, p_drop=0.1, margin=0.3):
        super().__init__()
        self.encoder = tower
        with torch.no_grad():                                   # measure tower width once
            d = run_tower(tower, torch.zeros(1, WIN)).shape[-1]
        self.head_embedding = nn.Linear(d, proj_dim)
        self.classifier = nn.Sequential(nn.Dropout(p_drop), nn.Linear(proj_dim, num_classes))
        self.margin = margin

    def embed(self, x):                                         # (B, WIN) -> (B, 128)
        return self.head_embedding(run_tower(self.encoder, x).mean(1).float())

    def forward(self, x, labels=None, positive=None, negative=None):
        has_triplet = positive is not None and negative is not None
        B = x.shape[0]
        z = self.embed(torch.cat([x, positive, negative]) if has_triplet else x)   # one encoder pass
        normed = F.normalize(z, dim=-1)
        anc, logits = normed[:B], self.classifier(z[:B])

        out = {"classifications": logits, "embeddings": anc}
        if labels is not None:
            cls_loss = F.cross_entropy(logits, labels)
            out["classification_loss"] = cls_loss
            out["total_loss"] = cls_loss
            if has_triplet:
                pos, neg = normed[B:2 * B], normed[2 * B:]
                trip = F.triplet_margin_loss(anc, pos, neg, margin=self.margin)
                out["triplet_loss"] = trip
                out["total_loss"] = cls_loss + 0.5 * trip
        return out

# ---- training: x, positive, negative are (B, WIN); labels (B,) long ----
def train_step(clf, opt, x, labels, positive, negative):
    clf.train()
    fwd = clf(x, labels=labels, positive=positive, negative=negative)
    fwd["total_loss"].backward(); opt.step(); opt.zero_grad()
    acc = (fwd["classifications"].argmax(-1) == labels).float().mean().item()
    return (fwd["total_loss"].item(), fwd["triplet_loss"].item(),
            fwd["classification_loss"].item(), acc)

clf = WDPModel(audio_tower, NUM_CLASSES).to(DEVICE)
opt = torch.optim.AdamW([
    {"params": [p for p in clf.encoder.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": clf.head_embedding.parameters(), "lr": 1e-3},
    {"params": clf.classifier.parameters(), "lr": 1e-3},
])

Loading weights:   0%|          | 0/1165 [00:00<?, ?it/s]

In [18]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA RTX PRO 6000 Blackwell Server Edition


In [19]:



AUDIO  = '/content/drive/MyDrive/dolphin data/l1.wav'
LABELS = '/content/drive/MyDrive/dolphin data/l1.csv'

def load_filtered_waveform(filename):
    waveform, sr = torchaudio.load(filename)
    if waveform.shape[0] > 1:
        # process()'s x.squeeze() only drops size-1 dims, so a multi-channel
        # waveform reaches the Whisper feature extractor as 2D instead of
        # the 1D it expects, breaking its internal padding. Every file this
        # pipeline was built/trained against happens to be mono already, so
        # this is a no-op there -- only multi-channel input hits it.
        waveform = waveform[:1]
    waveform = torchaudio.functional.highpass_biquad(
        waveform, sr, 5000
    )
    waveform = torchaudio.functional.lowpass_biquad(
        waveform, sr, 7900
    )
    return waveform, sr

def instance(label, offset, reverse, by_label, audio):
    negative_label = np.random.choice(list(all_labels.difference({label})))
    x_pos_offset   = np.random.choice(list(by_label[label].difference({offset})))
    x_neg_offset   = np.random.choice(list(by_label[negative_label]))

    y     = reverse[label]
    x     = audio[0, offset:offset + WIN]
    x_pos = audio[0, x_pos_offset:x_pos_offset + WIN]
    x_neg = audio[0, x_neg_offset:x_neg_offset + WIN]
    return x, x_pos, x_neg, y


def batch(instances, i, batch_size=100):
    x   = torch.stack([instances[i][0] for i in range(i, i + batch_size)])
    y   = torch.tensor([instances[i][3] for i in range(i, i + batch_size)])
    pos = torch.stack([instances[i][1] for i in range(i, i + batch_size)])
    neg = torch.stack([instances[i][2] for i in range(i, i + batch_size)])
    return x, pos, neg, y


In [20]:
LABEL_MAP = {
    2: 'WSTL_DOWN',
    1: 'WSTL_UP',
    4: 'BURST',
    3: 'ECHO',
    0: 'NOISE'
}
reverse    = {v:k for k, v in LABEL_MAP.items()}
all_labels = set(reverse.keys())
audio      = load_filtered_waveform(AUDIO)[0]
labels     = pd.read_csv(LABELS)
labels     = labels.sample(frac=1).reset_index()
by_label   = labels.groupby(labels.annotation)['offset'].apply(set)

labels['instance'] = labels.apply(lambda x: instance(x['annotation'], x['offset'], reverse, by_label, audio), axis=1)
instances          = list(labels['instance'])

In [ ]:
for batch_i in range(0, 25):
    complete = 0
    for i in range(0, len(instances) - 100, 100):
        x, pos, neg, y = batch(instances, i)
        y = y.to(DEVICE)

        total, classification, triplet, acc = train_step(clf, opt, x, y, pos, neg)
        complete += total
        print(f"{total} = {classification} {triplet} [{acc}]")
    print(f"====================== {batch_i} {complete} ===========================")

In [24]:
import shutil

OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2"
os.makedirs(OUT, exist_ok=True)
clf.encoder.save_pretrained(f"{OUT}/audio-lora")                 # LoRA adapter weights + adapter config
torch.save({"head_embedding": clf.head_embedding.state_dict(),
            "classifier":     clf.classifier.state_dict(),
            "config": {"base_model": MID, "sr": SR, "win": WIN, "num_classes": NUM_CLASSES,
                       "proj_dim": PROJ_DIM, "margin": clf.margin, "label_map": LABEL_MAP}},
           f"{OUT}/wdp-heads.pt")
print(shutil.make_archive(OUT, "zip", OUT))                       # single file to download

/content/drive/MyDrive/dolphin/wdp-embeddinggemma2.zip


In [ ]:
from peft import PeftModel

# ---- reload a saved model (run cell 2 first for MID / feature_extractor / run_tower / WDPModel) ----
OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2"
ck  = torch.load(f"{OUT}/wdp-heads.pt", map_location="cpu")

base = AutoModel.from_pretrained(MID, config=cfg, torch_dtype=torch.float32)
audio_tower = next(m for m in base.modules() if type(m).__name__.endswith("AudioModel"))
del base
audio_tower = PeftModel.from_pretrained(audio_tower, f"{OUT}/audio-lora", is_trainable=True)

clf = WDPModel(audio_tower, ck["config"]["num_classes"], ck["config"]["proj_dim"], margin=ck["config"]["margin"])
clf.head_embedding.load_state_dict(ck["head_embedding"])
clf.classifier.load_state_dict(ck["classifier"])
clf = clf.to(DEVICE)
opt = torch.optim.AdamW([                                          # fresh optimizer to continue training
    {"params": [p for p in clf.encoder.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": clf.head_embedding.parameters(), "lr": 1e-3},
    {"params": clf.classifier.parameters(), "lr": 1e-3},
])